# Pre-Requisites

## Load 'employee.csv' into DataFrame


In [0]:
df = spark.read.csv(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_001.csv",
    header=True,
    inferSchema=True,
)
df.limit(4).display()

# Write into Parquet

In [0]:
df.write.format("parquet").save(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_parquet",
    mode="overwrite",
)
df_parquet = spark.read.parquet(
    "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_parquet"
)
df_parquet.limit(4).display()

## Create a View


In [0]:
df_parquet.createOrReplaceTempView("user_vw")

# Drawback - Updates are not allowed 
- Parquet do not allow updates. Updates are only allowed in Delta format

In [0]:
%sql
UPDATE user_vw
SET country = 'Bharat'
WHERE country = 'India'

# Disadvantage - Overwrite
- Old files gets overwrites and the changes can't be rolled back.

In [0]:
from pyspark.sql.functions import col
df.filter(col("country")=="India").write.format("parquet").save(
    path="/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_parquet",
    mode="overwrite",
)

In [0]:
spark.read.parquet(
    "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_parquet"
).limit(6).display()

# Disadvantage - Data Quality Issue
- The existing file doesnot have an education column and when the new data with education column is overwrited, It gets overwritten without any issues.

In [0]:
spark.read.csv(
    path = "/Volumes/merit_catalog/quickstart_schema/sandbox/dataset/user_dataset/users_006_new_column_education.csv",
    header = True,
    inferSchema = True
).write.format("parquet").mode("append").save(
    path = "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_parquet"
)

In [0]:
spark.read.parquet(
    "/Volumes/merit_catalog/quickstart_schema/sandbox/output/output_parquet"
).limit(6).display()